# THỰC HÀNH LÀM SẠCH DỮ LIỆU VỚI PYSPARK

## Nội dung thực hành

1. Giải thích vai trò của data cleaning trong quy trình Big Data.
2. Định nghĩa schema rõ ràng khi đọc dữ liệu.
3. Hiểu tính immutable và lazy processing của Spark.
4. Đọc/ghi dữ liệu ở định dạng CSV và Parquet.
5. Thực hiện các phép biến đổi cột của DataFrame.
6. Lọc dữ liệu lỗi, dữ liệu null và dữ liệu bất thường.
7. Dùng `when()`, `otherwise()` và UDF.
8. Hiểu partition, ID phân tán và caching.
9. Quan sát execution plan, shuffle và broadcast join.
10. Xây dựng pipeline làm sạch dữ liệu hoàn chỉnh.

<a id="setup"></a>
## 1. Chuẩn bị môi trường

### 1.1. Cài đặt PySpark

In [1]:
import importlib.util
import subprocess
import sys

if importlib.util.find_spec("pyspark") is None:
    print("Chưa có PySpark. Đang cài đặt...")
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "pyspark"])
else:
    print("PySpark đã sẵn sàng.")

PySpark đã sẵn sàng.


### 1.2. Xác định thư mục dataset

In [2]:
from pathlib import Path

candidates = [
    Path.cwd() / "Datasets",
    Path("/content/Datasets"),
    Path.cwd().parent / "Datasets"
]

DATA_DIR = next((p for p in candidates if p.exists()), None)

if DATA_DIR is None:
    print("Không tìm thấy thư mục Datasets.")
else:
    print("DATA_DIR =", DATA_DIR.resolve())
    for p in sorted(DATA_DIR.iterdir()):
        print(" -", p.name)

DATA_DIR = C:\Users\chinhnt0500\Downloads\33241026114_NguyenThiChinh_CleanDataPySpark\Datasets
 - AA_DFW_2014_Departures_Short.csv.gz
 - AA_DFW_2015_Departures_Short.csv.gz
 - AA_DFW_2016_Departures_Short.csv.gz
 - AA_DFW_2017_Departures_Short.csv.gz
 - DallasCouncilVoters.csv.gz
 - DallasCouncilVotes.csv.gz


### 1.3. Khởi tạo SparkSession

In [3]:
from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.types import *

spark = (
    SparkSession.builder
    .appName("CleaningDataWithPySpark")
    .master("local[*]")
    .getOrCreate()
)

sc = spark.sparkContext

print("Spark version :", spark.version)
print("Master        :", sc.master)
print("Parallelism   :", sc.defaultParallelism)

Spark version : 3.5.3
Master        : local[*]
Parallelism   : 20


## 2. Giới thiệu Data Cleaning

Data Cleaning là quá trình chuẩn bị dữ liệu thô trước khi đưa vào pipeline xử lý.

Các công việc thường gặp:

- chuẩn hóa hoặc thay thế chuỗi;
- tính toán lại giá trị;
- loại dữ liệu rác;
- xử lý dữ liệu thiếu;
- kiểm tra kiểu dữ liệu;
- chuẩn hóa định dạng;
- xác thực dữ liệu.

Spark phù hợp cho data cleaning khi dữ liệu có kích thước lớn vì có khả năng xử lý phân tán và mở rộng.

### 2.1. Ví dụ làm sạch dữ liệu đơn giản

Dữ liệu thô:

| name | age (years) | city |
|---|---:|---|
| Smith, John | 37 | Dallas |
| Wilson, A. | 59 | Chicago |
| null | 215 | null |

Mục tiêu:

- tách họ và tên;
- đổi tuổi từ năm sang tháng;
- đổi tên thành phố sang mã bang;
- loại dòng không hợp lệ.

In [4]:
raw_df = spark.sql("""
    SELECT *
    FROM VALUES
        ('Smith, John', 37, 'Dallas'),
        ('Wilson, A.', 59, 'Chicago'),
        (CAST(NULL AS STRING), 215, CAST(NULL AS STRING))
    AS people(name, age_years, city)
""")

raw_df.show(truncate=False)

+-----------+---------+-------+
|name       |age_years|city   |
+-----------+---------+-------+
|Smith, John|37       |Dallas |
|Wilson, A. |59       |Chicago|
|NULL       |215      |NULL   |
+-----------+---------+-------+



In [5]:
city_to_state = (
    F.when(F.col("city") == "Dallas", "TX")
     .when(F.col("city") == "Chicago", "IL")
)

clean_demo_df = (
    raw_df
    .filter(F.col("name").isNotNull())
    .withColumn("last_name", F.trim(F.split("name", ",").getItem(0)))
    .withColumn("first_name", F.trim(F.split("name", ",").getItem(1)))
    .withColumn("age_months", F.col("age_years") * 12)
    .withColumn("state", city_to_state)
    .select("last_name", "first_name", "age_months", "state")
)

clean_demo_df.show(truncate=False)

+---------+----------+----------+-----+
|last_name|first_name|age_months|state|
+---------+----------+----------+-----+
|Smith    |John      |444       |TX   |
|Wilson   |A.        |708       |IL   |
+---------+----------+----------+-----+



<a id="schema"></a>
## 3. Spark Schema

Schema mô tả cấu trúc của DataFrame:

- tên cột;
- kiểu dữ liệu;
- có cho phép `null` hay không.

Việc khai báo schema rõ ràng giúp:

- phát hiện dữ liệu sai kiểu sớm hơn;
- tránh Spark phải quét dữ liệu nhiều lần để suy luận kiểu;
- tăng tính nhất quán trong pipeline.

### 3.1. Định nghĩa schema bằng `StructType`

In [6]:
people_schema = StructType([
    StructField("name", StringType(), False),
    StructField("age", IntegerType(), False),
    StructField("city", StringType(), False)
])

print(people_schema.simpleString())

struct<name:string,age:int,city:string>


### 3.2. Ví dụ schema cho dữ liệu chuyến bay

Dataset trong ZIP gồm dữ liệu chuyến bay American Airlines từ DFW.

In [7]:
flight_schema = StructType([
    StructField("Date (MM/DD/YYYY)", StringType(), True),
    StructField("Flight Number", StringType(), True),
    StructField("Destination Airport", StringType(), True),
    StructField("Actual elapsed time (Minutes)", IntegerType(), True)
])

In [8]:
if DATA_DIR is not None:
    flight_2014 = spark.read.csv(
        str(DATA_DIR / "AA_DFW_2014_Departures_Short.csv.gz"),
        header=True,
        schema=flight_schema
    )

    flight_2014.printSchema()
    flight_2014.show(5)

root
 |-- Date (MM/DD/YYYY): string (nullable = true)
 |-- Flight Number: string (nullable = true)
 |-- Destination Airport: string (nullable = true)
 |-- Actual elapsed time (Minutes): integer (nullable = true)

+-----------------+-------------+-------------------+-----------------------------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|
+-----------------+-------------+-------------------+-----------------------------+
|       01/01/2014|         0005|                HNL|                          519|
|       01/01/2014|         0007|                OGG|                          505|
|       01/01/2014|         0035|                SLC|                          174|
|       01/01/2014|         0043|                DTW|                          153|
|       01/01/2014|         0052|                PIT|                          137|
+-----------------+-------------+-------------------+-----------------------------+
only showing top 5 rows



### So sánh với `inferSchema=True`

`inferSchema=True` tiện lợi trong ví dụ nhỏ, nhưng trên dữ liệu lớn Spark phải đọc thêm dữ liệu để suy luận kiểu.

Trong các pipeline production, nên ưu tiên schema được định nghĩa trước nếu cấu trúc dữ liệu đã biết.

## 4. Immutability và Lazy Processing

### 4.1. Immutability

Spark DataFrame là immutable, không sửa trực tiếp DataFrame gốc.

Mỗi transformation tạo ra một DataFrame mới về mặt logic.

In [9]:
from pyspark.sql import functions as F

df_original = spark.sql("""
    SELECT *
    FROM VALUES
        ('Alice', 24),
        ('Bob', 30)
    AS people(name, age)
""")

# Thêm cột vào DataFrame mới
df_new = df_original.withColumn(
    "age_next_year",
    F.col("age") + 1
)

print("DataFrame gốc:")
df_original.show()

print("DataFrame mới:")
df_new.show()

DataFrame gốc:
+-----+---+
| name|age|
+-----+---+
|Alice| 24|
|  Bob| 30|
+-----+---+

DataFrame mới:
+-----+---+-------------+
| name|age|age_next_year|
+-----+---+-------------+
|Alice| 24|           25|
|  Bob| 30|           31|
+-----+---+-------------+



### 4.2. Lazy Processing

Các transformation như:

- `withColumn()`
- `select()`
- `filter()`
- `drop()`

chưa thực thi ngay.

Spark chỉ thật sự xử lý khi gặp **action** như:

- `show()`
- `count()`
- `collect()`
- `write`

In [10]:
if DATA_DIR is not None:
    lazy_df = spark.read.csv(
        str(DATA_DIR / "AA_DFW_2015_Departures_Short.csv.gz"),
        header=True,
        inferSchema=True
    )

    lazy_df = lazy_df.withColumn(
        "airport",
        F.lower(F.col("Destination Airport"))
    )

    lazy_df = lazy_df.drop("Destination Airport")

    # Action:
    lazy_df.show(5)

+-----------------+-------------+-----------------------------+-------+
|Date (MM/DD/YYYY)|Flight Number|Actual elapsed time (Minutes)|airport|
+-----------------+-------------+-----------------------------+-------+
|       01/01/2015|            5|                          526|    hnl|
|       01/01/2015|            7|                          517|    ogg|
|       01/01/2015|           23|                          233|    sfo|
|       01/01/2015|           27|                          165|    las|
|       01/01/2015|           29|                            0|    ont|
+-----------------+-------------+-----------------------------+-------+
only showing top 5 rows



### 4.3. Xem Logical/Physical Plan

`explain()` giúp quan sát kế hoạch Spark dự kiến thực hiện.

In [11]:
if DATA_DIR is not None:
    lazy_df.filter(F.col("airport") == "hnl").select("airport").explain()

== Physical Plan ==
*(1) Project [lower(Destination Airport#145) AS airport#151]
+- *(1) Filter (isnotnull(Destination Airport#145) AND (lower(Destination Airport#145) = hnl))
   +- FileScan csv [Destination Airport#145] Batched: false, DataFilters: [isnotnull(Destination Airport#145), (lower(Destination Airport#145) = hnl)], Format: CSV, Location: InMemoryFileIndex(1 paths)[file:/C:/Users/chinhnt0500/Downloads/33241026114_NguyenThiChinh_CleanD..., PartitionFilters: [], PushedFilters: [IsNotNull(Destination Airport)], ReadSchema: struct<Destination Airport:string>




## 5. Làm việc với Parquet

CSV dễ trao đổi nhưng có các hạn chế:

- không lưu schema;
- cần parse chuỗi;
- kém hiệu quả với dữ liệu lớn;
- hạn chế predicate pushdown.

Parquet là định dạng dạng cột (columnar), phù hợp với Spark vì:

- lưu schema;
- hỗ trợ predicate pushdown;
- đọc chọn lọc cột hiệu quả;
- thường tốt hơn cho phân tích.

### 5.1. Gộp nhiều năm dữ liệu

In [12]:
if DATA_DIR is not None:
    files = [
        "AA_DFW_2014_Departures_Short.csv.gz",
        "AA_DFW_2015_Departures_Short.csv.gz",
        "AA_DFW_2016_Departures_Short.csv.gz",
        "AA_DFW_2017_Departures_Short.csv.gz"
    ]

    flight_dfs = [
        spark.read.csv(str(DATA_DIR / f), header=True, schema=flight_schema)
        for f in files
    ]

    flights_all = flight_dfs[0]
    for df in flight_dfs[1:]:
        flights_all = flights_all.unionByName(df)

    print("Tổng số dòng:", flights_all.count())
    flights_all.show(5)

Tổng số dòng: 583718
+-----------------+-------------+-------------------+-----------------------------+
|Date (MM/DD/YYYY)|Flight Number|Destination Airport|Actual elapsed time (Minutes)|
+-----------------+-------------+-------------------+-----------------------------+
|       01/01/2014|         0005|                HNL|                          519|
|       01/01/2014|         0007|                OGG|                          505|
|       01/01/2014|         0035|                SLC|                          174|
|       01/01/2014|         0043|                DTW|                          153|
|       01/01/2014|         0052|                PIT|                          137|
+-----------------+-------------+-------------------+-----------------------------+
only showing top 5 rows



### 5.2. Ghi và đọc lại Parquet

In [13]:
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import pyarrow as pa
import pyarrow.parquet as pq

from pyspark.sql.pandas.types import to_arrow_schema


# 1. Kiểm tra DataFrame nguồn
if "flights_all" not in globals():
    raise NameError(
        "Chưa có flights_all. "
        "Hãy chạy lại ô đọc dữ liệu chuyến bay trước."
    )


# 2. Tạo thư mục mới cho mỗi lần chạy
folder_name = (
    f"AA_DFW_ALL_pyarrow_"
    f"{datetime.now():%Y%m%d_%H%M%S}_"
    f"{uuid4().hex[:6]}.parquet"
)

output_dir = Path.cwd() / folder_name
output_dir.mkdir(parents=True, exist_ok=False)

print("Thư mục lưu:", output_dir)


# 3. Chuẩn bị schema và kích thước batch
arrow_schema = to_arrow_schema(flights_all.schema)

batch_size = 20_000
batch = []
part_id = 0
total_written = 0


# 4. Hàm ghi một batch
def write_batch(rows, part_number):
    table = pa.Table.from_pylist(
        rows,
        schema=arrow_schema
    )

    file_path = output_dir / f"part-{part_number:05d}.parquet"

    pq.write_table(
        table,
        file_path,
        compression="snappy"
    )


# 5. Ghi lần lượt từng batch
# Không collect()/toPandas() toàn bộ dữ liệu
for row in flights_all.toLocalIterator():
    batch.append(row.asDict(recursive=True))

    if len(batch) >= batch_size:
        write_batch(batch, part_id)

        total_written += len(batch)
        part_id += 1
        batch.clear()


# Ghi phần dữ liệu còn lại
if batch:
    write_batch(batch, part_id)

    total_written += len(batch)
    part_id += 1
    batch.clear()


# Nếu dữ liệu rỗng, vẫn tạo file có schema
if part_id == 0:
    write_batch([], part_id)
    part_id += 1


print(f"Số file Parquet: {part_id:,}")
print(f"Số dòng đã ghi: {total_written:,}")


# 6. Kiểm tra dữ liệu đã ghi bằng PyArrow
import pyarrow.parquet as pq

parquet_files = sorted(output_dir.glob("part-*.parquet"))

if not parquet_files:
    raise FileNotFoundError(
        f"Không tìm thấy file Parquet trong: {output_dir}"
    )

# Đếm dòng từ metadata, không tải toàn bộ dữ liệu vào bộ nhớ
total_read = sum(
    pq.ParquetFile(file).metadata.num_rows
    for file in parquet_files
)

print(f"Số file Parquet: {len(parquet_files):,}")
print(f"Số dòng đã ghi: {total_written:,}")
print(f"Số dòng kiểm tra: {total_read:,}")

assert total_read == total_written, "Số dòng không khớp!"

# Hiển thị schema
first_file = pq.ParquetFile(parquet_files[0])

print("\nSchema Parquet:")
print(first_file.schema_arrow)

# Đọc tối đa 5 dòng từ file đầu tiên
print("\nDữ liệu mẫu:")

sample_batch = next(
    first_file.iter_batches(batch_size=5),
    None
)

if sample_batch is not None:
    for row in sample_batch.to_pylist():
        print(row)

print("\nĐã kiểm tra file tại:", output_dir)

# Tiếp tục phân tích bằng DataFrame Spark đang có
flights_analysis = flights_all

Thư mục lưu: C:\Users\chinhnt0500\Downloads\33241026114_NguyenThiChinh_CleanDataPySpark\AA_DFW_ALL_pyarrow_20261026_135850_ccd015.parquet
Số file Parquet: 30
Số dòng đã ghi: 583,718
Số file Parquet: 30
Số dòng đã ghi: 583,718
Số dòng kiểm tra: 583,718

Schema Parquet:
Date (MM/DD/YYYY): string
Flight Number: string
Destination Airport: string
Actual elapsed time (Minutes): int32

Dữ liệu mẫu:
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0005', 'Destination Airport': 'HNL', 'Actual elapsed time (Minutes)': 519}
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0007', 'Destination Airport': 'OGG', 'Actual elapsed time (Minutes)': 505}
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0035', 'Destination Airport': 'SLC', 'Actual elapsed time (Minutes)': 174}
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0043', 'Destination Airport': 'DTW', 'Actual elapsed time (Minutes)': 153}
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0052', 'Destination Airport': 'PIT

In [14]:
from pathlib import Path
from datetime import datetime
from uuid import uuid4

import pyarrow as pa
import pyarrow.parquet as pq
from pyspark.sql.pandas.types import to_arrow_schema


if DATA_DIR is not None:

    if "flights_all" not in globals():
        raise NameError(
            "Chưa có flights_all. "
            "Hãy chạy ô đọc dữ liệu chuyến bay trước."
        )

    # 1. Tạo thư mục riêng cho mỗi lần chạy
    folder_name = (
        f"AA_DFW_ALL_pyarrow_"
        f"{datetime.now():%Y%m%d_%H%M%S}_"
        f"{uuid4().hex[:6]}.parquet"
    )

    output_dir = Path.cwd() / folder_name
    output_dir.mkdir(parents=True, exist_ok=False)

    PARQUET_PATH = str(output_dir)

    # 2. Chuẩn bị schema
    arrow_schema = to_arrow_schema(flights_all.schema)

    batch_size = 20_000
    batch = []
    part_id = 0
    total_written = 0

    def write_batch(rows, part_number):
        table = pa.Table.from_pylist(
            rows,
            schema=arrow_schema
        )

        pq.write_table(
            table,
            output_dir / f"part-{part_number:05d}.parquet",
            compression="snappy"
        )

    # 3. Ghi dữ liệu từng batch
    # Không collect hoặc toPandas toàn bộ dữ liệu
    for row in flights_all.toLocalIterator():
        batch.append(row.asDict(recursive=True))

        if len(batch) >= batch_size:
            write_batch(batch, part_id)

            total_written += len(batch)
            part_id += 1
            batch.clear()

    if batch:
        write_batch(batch, part_id)

        total_written += len(batch)
        part_id += 1
        batch.clear()

    # Dữ liệu rỗng vẫn ghi file có schema
    if part_id == 0:
        write_batch([], part_id)

    # 4. Kiểm tra số dòng từ metadata Parquet
    parquet_files = sorted(output_dir.glob("part-*.parquet"))

    total_read = sum(
        pq.ParquetFile(file).metadata.num_rows
        for file in parquet_files
    )

    print("Đường dẫn:", PARQUET_PATH)
    print(f"Số file Parquet: {len(parquet_files):,}")
    print(f"Số dòng đã ghi: {total_written:,}")
    print(f"Số dòng trong metadata: {total_read:,}")

    assert total_read == total_written, "Số dòng không khớp!"

    # 5. Hiển thị schema và tối đa 5 dòng từ file đầu
    first_file = pq.ParquetFile(parquet_files[0])

    print("\nSchema Parquet:")
    print(first_file.schema_arrow)

    print("\nDữ liệu mẫu:")
    sample_batch = next(
        first_file.iter_batches(batch_size=5),
        None
    )

    if sample_batch is not None:
        for record in sample_batch.to_pylist():
            print(record)

    # Tiếp tục phân tích bằng DataFrame Spark nguồn
    flights_analysis = flights_all

    print("\nHoàn tất ghi và kiểm tra bằng PyArrow.")

else:
    print("DATA_DIR chưa được thiết lập.")

Đường dẫn: C:\Users\chinhnt0500\Downloads\33241026114_NguyenThiChinh_CleanDataPySpark\AA_DFW_ALL_pyarrow_20261026_135855_5723f7.parquet
Số file Parquet: 30
Số dòng đã ghi: 583,718
Số dòng trong metadata: 583,718

Schema Parquet:
Date (MM/DD/YYYY): string
Flight Number: string
Destination Airport: string
Actual elapsed time (Minutes): int32

Dữ liệu mẫu:
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0005', 'Destination Airport': 'HNL', 'Actual elapsed time (Minutes)': 519}
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0007', 'Destination Airport': 'OGG', 'Actual elapsed time (Minutes)': 505}
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0035', 'Destination Airport': 'SLC', 'Actual elapsed time (Minutes)': 174}
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0043', 'Destination Airport': 'DTW', 'Actual elapsed time (Minutes)': 153}
{'Date (MM/DD/YYYY)': '01/01/2014', 'Flight Number': '0052', 'Destination Airport': 'PIT', 'Actual elapsed time (Minutes)': 137}

### 5.3. Parquet và Spark SQL

In [15]:
if DATA_DIR is not None:

    # Đăng ký DataFrame đang có thành bảng tạm
    flights_analysis.createOrReplaceTempView("flights")

    spark.sql("""
        SELECT
            `Destination Airport` AS airport,
            AVG(`Actual elapsed time (Minutes)`) AS avg_minutes,
            COUNT(*) AS n
        FROM flights
        GROUP BY `Destination Airport`
        ORDER BY n DESC
        LIMIT 10
    """).show(truncate=False)

+-------+------------------+-----+
|airport|avg_minutes       |n    |
+-------+------------------+-----+
|LAX    |195.81170700980834|22226|
|ORD    |134.2056630529253 |21861|
|SAT    |63.446708969670894|18596|
|AUS    |56.0158876979161  |18379|
|LGA    |191.42767295597486|17013|
|MIA    |161.56616822429908|16050|
|LAS    |170.62809139784946|14880|
|SFO    |222.81960865657194|14463|
|PHX    |149.9250647804468 |14279|
|MCO    |148.37622832369942|13840|
+-------+------------------+-----+



## 6. DataFrame Column Operations

Các transformation phổ biến:

- `filter()` / `where()`
- `select()`
- `withColumn()`
- `drop()`
- `withColumnRenamed()`

### 6.1. Dataset Dallas Council Voters

In [16]:
if DATA_DIR is not None:
    voter_df = spark.read.csv(
        str(DATA_DIR / "DallasCouncilVoters.csv.gz"),
        header=True,
        inferSchema=True
    )

    voter_df.printSchema()
    voter_df.show(10, truncate=False)

root
 |-- DATE: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER_NAME: string (nullable = true)

+----------+-------------+-------------------+
|DATE      |TITLE        |VOTER_NAME         |
+----------+-------------+-------------------+
|02/08/2017|Councilmember|Jennifer S. Gates  |
|02/08/2017|Councilmember|Philip T. Kingston |
|02/08/2017|Mayor        |Michael S. Rawlings|
|02/08/2017|Councilmember|Adam Medrano       |
|02/08/2017|Councilmember|Casey Thomas       |
|02/08/2017|Councilmember|Carolyn King Arnold|
|02/08/2017|Councilmember|Scott Griggs       |
|02/08/2017|Councilmember|B. Adam  McGough   |
|02/08/2017|Councilmember|Lee Kleinman       |
|02/08/2017|Councilmember|Sandy Greyson      |
+----------+-------------+-------------------+
only showing top 10 rows



### 6.2. Filtering

In [17]:
if DATA_DIR is not None:
    voter_df.select("VOTER_NAME").distinct().show(20, truncate=False)

    voter_clean = (
        voter_df
        .filter(F.col("VOTER_NAME").isNotNull())
        .filter(F.length("VOTER_NAME").between(1, 20))
        .filter(~F.col("VOTER_NAME").contains("_"))
    )

    voter_clean.select("VOTER_NAME").distinct().show(20, truncate=False)

+--------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+
|VOTER_NAME                                                                                                                                                                                                                                                                                                                                                                                                                |
+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------

### 6.3. String transformations

Các hàm thường dùng trong `pyspark.sql.functions`:

- `upper()`, `lower()`
- `trim()`
- `split()`
- `length()`
- `contains()`
- `regexp_replace()`

In [18]:
if DATA_DIR is not None:
    voter_string = (
        voter_df
        .filter(F.col("VOTER_NAME").isNotNull())
        .withColumn("NAME_UPPER", F.upper("VOTER_NAME"))
        .withColumn("NAME_PARTS", F.split("VOTER_NAME", " "))
        .withColumn("PART_COUNT", F.size("NAME_PARTS"))
    )

    voter_string.select(
        "VOTER_NAME", "NAME_UPPER", "NAME_PARTS", "PART_COUNT"
    ).show(10, truncate=False)

+-------------------+-------------------+-----------------------+----------+
|VOTER_NAME         |NAME_UPPER         |NAME_PARTS             |PART_COUNT|
+-------------------+-------------------+-----------------------+----------+
|Jennifer S. Gates  |JENNIFER S. GATES  |[Jennifer, S., Gates]  |3         |
|Philip T. Kingston |PHILIP T. KINGSTON |[Philip, T., Kingston] |3         |
|Michael S. Rawlings|MICHAEL S. RAWLINGS|[Michael, S., Rawlings]|3         |
|Adam Medrano       |ADAM MEDRANO       |[Adam, Medrano]        |2         |
|Casey Thomas       |CASEY THOMAS       |[Casey, Thomas]        |2         |
|Carolyn King Arnold|CAROLYN KING ARNOLD|[Carolyn, King, Arnold]|3         |
|Scott Griggs       |SCOTT GRIGGS       |[Scott, Griggs]        |2         |
|B. Adam  McGough   |B. ADAM  MCGOUGH   |[B., Adam, , McGough]  |4         |
|Lee Kleinman       |LEE KLEINMAN       |[Lee, Kleinman]        |2         |
|Sandy Greyson      |SANDY GREYSON      |[Sandy, Greyson]       |2         |

### 6.4. Tách first name / last name

[MỞ RỘNG] Code trong ZIP tách tên từ dữ liệu Dallas Council.

In [19]:
if DATA_DIR is not None:
    voter_names = (
        voter_df
        .filter(F.col("VOTER_NAME").isNotNull())
        .withColumn("name_parts", F.split(F.trim("VOTER_NAME"), " "))
        .withColumn("first_name", F.col("name_parts").getItem(0))
        .withColumn(
            "last_name",
            F.col("name_parts").getItem(
                F.size("name_parts") - 1
            )
        )
    )

    voter_names.select(
        "VOTER_NAME", "first_name", "last_name"
    ).show(10, truncate=False)

+-------------------+----------+---------+
|VOTER_NAME         |first_name|last_name|
+-------------------+----------+---------+
|Jennifer S. Gates  |Jennifer  |Gates    |
|Philip T. Kingston |Philip    |Kingston |
|Michael S. Rawlings|Michael   |Rawlings |
|Adam Medrano       |Adam      |Medrano  |
|Casey Thomas       |Casey     |Thomas   |
|Carolyn King Arnold|Carolyn   |Arnold   |
|Scott Griggs       |Scott     |Griggs   |
|B. Adam  McGough   |B.        |McGough  |
|Lee Kleinman       |Lee       |Kleinman |
|Sandy Greyson      |Sandy     |Greyson  |
+-------------------+----------+---------+
only showing top 10 rows



C:\Users\chinhnt0500\AppData\Local\anaconda3\envs\pyspark_bt01\Lib\site-packages\pyspark\sql\column.py:460: FutureWarning: A column as 'key' in getItem is deprecated as of Spark 3.0, and will not be supported in the future release. Use `column[key]` or `column.key` syntax instead.
  warnings.warn(


## 7. Conditional DataFrame Column Operations

Spark hỗ trợ logic tương tự `if / elif / else` qua:

- `F.when()`
- `.otherwise()`

In [20]:
from pyspark.sql import functions as F

age_df = spark.sql("""
    SELECT *
    FROM VALUES
        ('Alice', 14),
        ('Bob', 18),
        ('Candice', 38)
    AS people(Name, Age)
""")

classified_df = age_df.withColumn(
    "Age_Group",
    F.when(F.col("Age") >= 18, "Adult")
     .otherwise("Minor")
)

classified_df.show()

+-------+---+---------+
|   Name|Age|Age_Group|
+-------+---+---------+
|  Alice| 14|    Minor|
|    Bob| 18|    Adult|
|Candice| 38|    Adult|
+-------+---+---------+



### 7.1. Nhiều điều kiện

In [21]:
multi_df = age_df.withColumn(
    "Category",
    F.when(F.col("Age") < 13, "Child")
     .when(F.col("Age") < 18, "Teen")
     .otherwise("Adult")
)

multi_df.show()

+-------+---+--------+
|   Name|Age|Category|
+-------+---+--------+
|  Alice| 14|    Teen|
|    Bob| 18|   Adult|
|Candice| 38|   Adult|
+-------+---+--------+



## 8. User Defined Functions - UDF

UDF cho phép dùng hàm Python tùy biến trong DataFrame.

Quy trình:

1. viết hàm Python;
2. bọc hàm bằng `F.udf`;
3. khai báo kiểu dữ liệu trả về;
4. dùng như một hàm Spark.

> Trong thực tế, nếu có thể biểu diễn bằng built-in Spark functions thì nên ưu tiên built-in functions vì thường tối ưu hơn Python UDF.

### 8.1. UDF đảo chuỗi

In [22]:
from pyspark.sql import functions as F

if DATA_DIR is not None:
    voter_udf = voter_df.withColumn(
        "ReverseName",
        F.reverse(F.col("VOTER_NAME"))
    )

    voter_udf.select(
        "VOTER_NAME",
        "ReverseName"
    ).show(10, truncate=False)

+-------------------+-------------------+
|VOTER_NAME         |ReverseName        |
+-------------------+-------------------+
|Jennifer S. Gates  |setaG .S refinneJ  |
|Philip T. Kingston |notsgniK .T pilihP |
|Michael S. Rawlings|sgnilwaR .S leahciM|
|Adam Medrano       |onardeM madA       |
|Casey Thomas       |samohT yesaC       |
|Carolyn King Arnold|dlonrA gniK nyloraC|
|Scott Griggs       |sggirG ttocS       |
|B. Adam  McGough   |hguoGcM  madA .B   |
|Lee Kleinman       |namnielK eeL       |
|Sandy Greyson      |nosyerG ydnaS      |
+-------------------+-------------------+
only showing top 10 rows



## 8.2. [MỞ RỘNG] UDF tính toán

In [23]:
from pyspark.sql import functions as F

if DATA_DIR is not None:

    name_words = F.filter(
        F.split(
            F.coalesce(F.col("VOTER_NAME"), F.lit("")),
            r"\s+"
        ),
        lambda word: word != ""
    )

    voter_udf = (
        voter_df
        .withColumn(
            "ReverseName",
            F.reverse(F.col("VOTER_NAME"))
        )
        .withColumn(
            "NameWordCount",
            F.size(name_words)
        )
    )

    voter_udf.select(
        "VOTER_NAME",
        "NameWordCount"
    ).show(10, truncate=False)

+-------------------+-------------+
|VOTER_NAME         |NameWordCount|
+-------------------+-------------+
|Jennifer S. Gates  |3            |
|Philip T. Kingston |3            |
|Michael S. Rawlings|3            |
|Adam Medrano       |2            |
|Casey Thomas       |2            |
|Carolyn King Arnold|3            |
|Scott Griggs       |2            |
|B. Adam  McGough   |3            |
|Lee Kleinman       |2            |
|Sandy Greyson      |2            |
+-------------------+-------------+
only showing top 10 rows



<a id="partition"></a>
# 9. Partitioning và ID

Spark chia DataFrame thành nhiều **partition**.  
Mỗi partition có thể được xử lý độc lập trên các worker.

Điều này giúp xử lý song song, nhưng cũng khiến việc tạo ID tuần tự `0,1,2,...` không tự nhiên trong môi trường phân tán.

## 9.1. Quan sát số partition

In [24]:
if DATA_DIR is not None:
    print("Partitions:", voter_df.rdd.getNumPartitions())

Partitions: 1


## 9.2. `monotonically_increasing_id()`

Hàm này tạo ID:

- 64-bit;
- duy nhất;
- tăng dần về mặt giá trị;
- **không đảm bảo liên tục**.

In [25]:
if DATA_DIR is not None:
    voter_with_id = voter_df.withColumn(
        "ID",
        F.monotonically_increasing_id()
    )

    voter_with_id.select("ID", "VOTER_NAME").show(10, truncate=False)

+---+-------------------+
|ID |VOTER_NAME         |
+---+-------------------+
|0  |Jennifer S. Gates  |
|1  |Philip T. Kingston |
|2  |Michael S. Rawlings|
|3  |Adam Medrano       |
|4  |Casey Thomas       |
|5  |Carolyn King Arnold|
|6  |Scott Griggs       |
|7  |B. Adam  McGough   |
|8  |Lee Kleinman       |
|9  |Sandy Greyson      |
+---+-------------------+
only showing top 10 rows



## 9.3. ID và Lazy Processing

Vì Spark lazy và partition-aware, ID có thể thay đổi khi kế hoạch thực thi thay đổi.  
Không nên xem `monotonically_increasing_id()` như primary key tuần tự của database.

## 10. Caching

Caching lưu DataFrame ở memory/disk để tái sử dụng nhanh hơn.

Phù hợp khi:

- một DataFrame được dùng nhiều lần;
- các bước trước đó tốn nhiều chi phí;
- dữ liệu đủ phù hợp với tài nguyên hệ thống.

Không nên cache mọi thứ.

### 10.1. Cache DataFrame

In [26]:
if DATA_DIR is not None:
    cached_df = voter_df.cache()

    # Action để materialize cache
    row_count = cached_df.count()

    print("Rows:", row_count)
    print("Is cached:", cached_df.is_cached)

Rows: 44625
Is cached: True


### 10.2. Unpersist

In [27]:
if DATA_DIR is not None:
    cached_df.unpersist()
    print("Is cached:", cached_df.is_cached)

Is cached: False


## 11. Cải thiện hiệu năng import

Một số nguyên tắc trong slide:

- định nghĩa schema rõ ràng;
- dữ liệu nên được chia thành các file có kích thước tương đối cân bằng;
- có thể đọc nhiều file bằng wildcard;
- dữ liệu trung gian nên chuyển sang Parquet nếu phù hợp.

### 11.1. Đọc nhiều file chuyến bay bằng wildcard

In [ ]:
from pathlib import Path

if DATA_DIR is not None:
    data_dir = Path(DATA_DIR)

    # Kiểm tra các file khớp wildcard trước khi đọc
    flight_files = sorted(
        data_dir.glob("AA_DFW_*_Departures_Short.csv.gz")
    )

    if not flight_files:
        raise FileNotFoundError(
            f"Không tìm thấy file chuyến bay trong: {data_dir}"
        )

    print(f"Tìm thấy {len(flight_files)} file:")
    for file in flight_files:
        print(" -", file.name)

    # Đọc bằng schema đã khai báo ở phần trước
    wildcard_path = str(
        data_dir / "AA_DFW_*_Departures_Short.csv.gz"
    )

    flights_wildcard = spark.read.csv(
        wildcard_path,
        header=True,
        schema=flight_schema
    )

    print("\nSchema:")
    flights_wildcard.printSchema()

    print("5 dòng dữ liệu mẫu:")
    flights_wildcard.show(5, truncate=False)

else:
    print("Chưa tìm thấy DATA_DIR. Hãy chạy cell xác định thư mục dữ liệu.")

Tìm thấy 4 file:
 - AA_DFW_2014_Departures_Short.csv.gz
 - AA_DFW_2015_Departures_Short.csv.gz
 - AA_DFW_2016_Departures_Short.csv.gz
 - AA_DFW_2017_Departures_Short.csv.gz


### 11.2. So sánh schema explicit và inferSchema

In [ ]:
if DATA_DIR is not None:
    explicit_df = spark.read.csv(
        wildcard_path,
        header=True,
        schema=flight_schema
    )

    inferred_df = spark.read.csv(
        wildcard_path,
        header=True,
        inferSchema=True
    )

    print("Explicit schema:")
    explicit_df.printSchema()

    print("Inferred schema:")
    inferred_df.printSchema()

### 11.3 Spark Configuration và Cluster

Spark có nhiều cấu hình có thể đọc/điều chỉnh qua:

```python
spark.conf.get(...)
spark.conf.set(...)
```

Trong notebook local, ta chỉ quan sát một số cấu hình an toàn.

In [ ]:
print("spark.sql.shuffle.partitions =", spark.conf.get("spark.sql.shuffle.partitions"))
print("spark.sql.adaptive.enabled   =", spark.conf.get("spark.sql.adaptive.enabled"))

### Kiến trúc cluster

Hai nhóm process quan trọng:

### Driver
- lập kế hoạch công việc;
- phân phối task;
- tổng hợp kết quả.

### Worker
- thực thi task;
- xử lý partition;
- đọc/ghi dữ liệu.

## 12. Execution Plan, Shuffle và Broadcast

### 12.1. Execution Plan

`explain()` giúp xem Spark sẽ thực hiện truy vấn như thế nào.

In [ ]:
if DATA_DIR is not None:
    voter_df.select("VOTER_NAME").distinct().explain()

### 12.2. Shuffle

Shuffle xảy ra khi Spark phải di chuyển dữ liệu giữa các partition/worker.

Các thao tác dễ phát sinh shuffle:

- `groupBy`
- `distinct`
- `join`
- `repartition`

Shuffle thường tốn network và I/O.

### 12.3. `coalesce()` và `repartition()`

- `repartition(n)`: có thể tăng/giảm partition, thường gây shuffle.
- `coalesce(n)`: chủ yếu dùng để giảm số partition với ít shuffle hơn.

In [ ]:
if DATA_DIR is not None:
    print("Ban đầu:", flights_wildcard.rdd.getNumPartitions())

    repart_df = flights_wildcard.repartition(4)
    print("repartition(4):", repart_df.rdd.getNumPartitions())

    coalesce_df = repart_df.coalesce(2)
    print("coalesce(2):", coalesce_df.rdd.getNumPartitions())

## 12.4. Broadcast Join

Nếu một DataFrame nhỏ, Spark có thể gửi bản sao tới các worker để giảm shuffle.

In [33]:
airport_info = spark.createDataFrame([
    ("HNL", "Honolulu"),
    ("OGG", "Kahului"),
    ("SFO", "San Francisco"),
    ("LAS", "Las Vegas"),
    ("DTW", "Detroit")
], ["airport_code", "airport_name"])

if DATA_DIR is not None:
    joined_broadcast = flights_wildcard.join(
        F.broadcast(airport_info),
        flights_wildcard["Destination Airport"] == airport_info["airport_code"],
        "left"
    )

    joined_broadcast.select(
        "Destination Airport", "airport_name"
    ).show(10)

+-------------------+------------+
|Destination Airport|airport_name|
+-------------------+------------+
|                HNL|    Honolulu|
|                OGG|     Kahului|
|                SLC|        NULL|
|                DTW|     Detroit|
|                PIT|        NULL|
|                SAN|        NULL|
|                MIA|        NULL|
|                JFK|        NULL|
|                ORD|        NULL|
|                STL|        NULL|
+-------------------+------------+
only showing top 10 rows


## 13. Data Pipeline

Một data pipeline gồm:

```text
Input
  ↓
Validation
  ↓
Cleaning
  ↓
Transformation
  ↓
Analysis
  ↓
Output
```

Nguồn có thể là:

- CSV
- JSON
- Parquet
- Database
- Web service

Output có thể là:

- Parquet
- CSV
- Database
- Dashboard / BI

### 13.1. Pipeline mẫu cho Dallas Council Votes

Ta sẽ:

1. đọc dữ liệu;
2. chuẩn hóa tên cột;
3. loại voter null;
4. chuẩn hóa chữ;
5. thêm ID;
6. tạo bảng phân tích.

In [34]:
if DATA_DIR is not None:
    votes_df = spark.read.csv(
        str(DATA_DIR / "DallasCouncilVotes.csv.gz"),
        header=True,
        inferSchema=True
    )

    votes_df.printSchema()

root
 |-- DATE: string (nullable = true)
 |-- AGENDA_ITEM_NUMBER: string (nullable = true)
 |-- ITEM_TYPE: string (nullable = true)
 |-- DISTRICT: string (nullable = true)
 |-- TITLE: string (nullable = true)
 |-- VOTER NAME: string (nullable = true)
 |-- VOTE CAST: string (nullable = true)
 |-- FINAL ACTION TAKEN: string (nullable = true)
 |-- AGENDA ITEM DESCRIPTION: string (nullable = true)
 |-- AGENDA_ID: string (nullable = true)
 |-- VOTE_ID: string (nullable = true)



In [35]:
if DATA_DIR is not None:
    votes_clean = (
        votes_df
        .withColumnRenamed("VOTER NAME", "VOTER_NAME")
        .withColumnRenamed("VOTE CAST", "VOTE_CAST")
        .filter(F.col("VOTER_NAME").isNotNull())
        .withColumn("VOTER_NAME", F.trim(F.col("VOTER_NAME")))
        .withColumn("VOTER_NAME_UPPER", F.upper("VOTER_NAME"))
        .withColumn("ROW_ID", F.monotonically_increasing_id())
    )

    votes_clean.select(
        "ROW_ID", "DATE", "VOTER_NAME", "VOTE_CAST"
    ).show(10, truncate=False)

+------+----------+-------------------+---------+
|ROW_ID|DATE      |VOTER_NAME         |VOTE_CAST|
+------+----------+-------------------+---------+
|0     |02/08/2017|Jennifer S. Gates  |N/A      |
|1     |02/08/2017|Philip T. Kingston |N/A      |
|2     |02/08/2017|Michael S. Rawlings|N/A      |
|3     |02/08/2017|Adam Medrano       |N/A      |
|4     |02/08/2017|Casey Thomas, II   |N/A      |
|5     |02/08/2017|Carolyn King Arnold|N/A      |
|6     |02/08/2017|Scott Griggs       |YES      |
|7     |02/08/2017|B. Adam  McGough   |YES      |
|8     |02/08/2017|Lee Kleinman       |YES      |
|9     |02/08/2017|Sandy Greyson      |YES      |
+------+----------+-------------------+---------+
only showing top 10 rows


### 13.2. Phân tích sau cleaning

In [36]:
if DATA_DIR is not None:
    (
        votes_clean
        .groupBy("VOTE_CAST")
        .count()
        .orderBy(F.desc("count"))
        .show(truncate=False)
    )

+-----------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+-----+
|VOTE_CAST                                                                                                                                                                                                                                        

### 13.3. Xuất Parquet

In [37]:
if DATA_DIR is not None:
    OUTPUT_PATH = str(Path.cwd() / "DallasCouncilVotes_Clean.parquet")

    (
        votes_clean.write
        .mode("overwrite")
        .parquet(OUTPUT_PATH)
    )

    print("Đã ghi:", OUTPUT_PATH)

Đã ghi: /content/DallasCouncilVotes_Clean.parquet


## 14. Data Handling và Validation

Dữ liệu CSV thực tế có thể chứa:

- dòng trống;
- comment;
- header;
- số cột không đồng nhất;
- nhiều delimiter;
- cấu trúc lồng nhau;
- dữ liệu sai kiểu.

Spark hỗ trợ nhiều tùy chọn khi đọc CSV.

### Ví dụ comment và separator

In [38]:
from tempfile import TemporaryDirectory
from pathlib import Path

tmp_dir = TemporaryDirectory()
sample_file = Path(tmp_dir.name) / "sample.csv"

sample_file.write_text(
    "# Dòng comment\n"
    "name;age;city\n"
    "Alice;23;HCMC\n"
    "Bob;31;Hanoi\n",
    encoding="utf-8"
)

sample_df = spark.read.csv(
    str(sample_file),
    header=True,
    sep=";",
    comment="#",
    inferSchema=True
)

sample_df.show()

+-----+---+-----+
| name|age| city|
+-----+---+-----+
|Alice| 23| HCMC|
|  Bob| 31|Hanoi|
+-----+---+-----+



### Data Validation

Validation là kiểm tra dữ liệu có phù hợp với format/rule mong đợi không.

Có thể kiểm tra:

- số dòng;
- số cột;
- kiểu dữ liệu;
- miền giá trị;
- khóa hợp lệ;
- quan hệ với bảng tham chiếu.

### 14.2. Validation bằng join

In [ ]:
valid_titles = spark.createDataFrame([
    ("Mayor",),
    ("Councilmember",)
], ["TITLE"])

if DATA_DIR is not None:
    before = voter_df.count()

    verified_voters = voter_df.join(
        F.broadcast(valid_titles),
        on="TITLE",
        how="inner"
    )

    after = verified_voters.count()

    print("Before:", before)
    print("After :", after)

### 14.2. Tìm dữ liệu không hợp lệ bằng `left_anti`

In [ ]:
if DATA_DIR is not None:
    invalid_voters = voter_df.join(
        F.broadcast(valid_titles),
        on="TITLE",
        how="left_anti"
    )

    print("Invalid rows:", invalid_voters.count())
    invalid_voters.show(10, truncate=False)

## 14.3. [MỞ RỘNG] Pipeline xử lý annotation dạng không chuẩn

Code ZIP có một ví dụ mở rộng với Stanford ImageNet dog annotations:

- tách nhiều trường từ một dòng;
- xử lý `ArrayType`;
- dùng UDF để chuyển chuỗi thành cấu trúc;
- validate folder bằng join;
- dùng `left_anti` để tìm dòng invalid;
- tính phần trăm pixel thuộc bounding box của chó.

**Dataset annotation không được đóng gói trong ZIP**, nên phần dưới dùng dữ liệu mô phỏng nhỏ để minh họa đúng kỹ thuật.

In [39]:
annotation_data = [
    ("02111277 n02111277_3206 500 375 Newfoundland,110,73,416,298",),
    ("02108422 n02108422_4375 500 375 bull_mastiff,101,90,214,356 bull_mastiff,282,74,416,370",)
]

annotations_df = spark.createDataFrame(annotation_data, ["raw"])
annotations_df.show(truncate=False)

+---------------------------------------------------------------------------------------+
|raw                                                                                    |
+---------------------------------------------------------------------------------------+
|02111277 n02111277_3206 500 375 Newfoundland,110,73,416,298                            |
|02108422 n02108422_4375 500 375 bull_mastiff,101,90,214,356 bull_mastiff,282,74,416,370|
+---------------------------------------------------------------------------------------+



In [40]:
split_cols = F.split(F.col("raw"), " ")

parsed = (
    annotations_df
    .withColumn("folder", split_cols.getItem(0))
    .withColumn("filename", split_cols.getItem(1))
    .withColumn("width", split_cols.getItem(2).cast("int"))
    .withColumn("height", split_cols.getItem(3).cast("int"))
    .withColumn("parts", split_cols)
)

parsed.select("folder", "filename", "width", "height", "parts").show(truncate=False)

+--------+--------------+-----+------+----------------------------------------------------------------------------------------------+
|folder  |filename      |width|height|parts                                                                                         |
+--------+--------------+-----+------+----------------------------------------------------------------------------------------------+
|02111277|n02111277_3206|500  |375   |[02111277, n02111277_3206, 500, 375, Newfoundland,110,73,416,298]                             |
|02108422|n02108422_4375|500  |375   |[02108422, n02108422_4375, 500, 375, bull_mastiff,101,90,214,356, bull_mastiff,282,74,416,370]|
+--------+--------------+-----+------+----------------------------------------------------------------------------------------------+



## 14. Bài tập mở rộng: Làm sạch dữ liệu chuyến bay

### Yêu cầu

Từ 4 file chuyến bay 2014-2017:

1. đọc bằng schema cố định;
2. gộp dữ liệu;
3. chuẩn hóa tên cột;
4. loại thời gian bay null hoặc <= 0;
5. thêm cột `year`;
6. chuẩn hóa airport thành uppercase;
7. tính thống kê theo năm và destination;
8. lưu kết quả thành Parquet.

In [ ]:
if DATA_DIR is not None:
    clean_flights = (
        flights_wildcard
        .withColumnRenamed("Date (MM/DD/YYYY)", "flight_date")
        .withColumnRenamed("Flight Number", "flight_number")
        .withColumnRenamed("Destination Airport", "destination")
        .withColumnRenamed(
            "Actual elapsed time (Minutes)",
            "elapsed_minutes"
        )
        .filter(F.col("elapsed_minutes").isNotNull())
        .filter(F.col("elapsed_minutes") > 0)
        .withColumn("destination", F.upper(F.trim("destination")))
        .withColumn(
            "flight_date_parsed",
            F.to_date("flight_date", "MM/dd/yyyy")
        )
        .withColumn("year", F.year("flight_date_parsed"))
    )

    clean_flights.printSchema()
    clean_flights.show(10)

In [42]:
if DATA_DIR is not None:
    flight_stats = (
        clean_flights
        .groupBy("year", "destination")
        .agg(
            F.count("*").alias("num_flights"),
            F.round(F.avg("elapsed_minutes"), 2).alias("avg_minutes"),
            F.min("elapsed_minutes").alias("min_minutes"),
            F.max("elapsed_minutes").alias("max_minutes")
        )
        .orderBy("year", F.desc("num_flights"))
    )

    flight_stats.show(30)

+----+-----------+-----------+-----------+-----------+-----------+
|year|destination|num_flights|avg_minutes|min_minutes|max_minutes|
+----+-----------+-----------+-----------+-----------+-----------+
|2014|        LAX|       5990|     195.25|        152|        331|
|2014|        ORD|       5790|     134.67|        105|        274|
|2014|        SAT|       5147|       63.4|         45|        178|
|2014|        AUS|       4965|      54.73|         40|        165|
|2014|        LGA|       4544|     194.32|        157|        327|
|2014|        LAS|       4034|     172.16|        144|        312|
|2014|        MIA|       3914|      160.7|        135|        303|
|2014|        SFO|       3796|     225.02|        191|        334|
|2014|        MCO|       3655|     146.87|        124|        267|
|2014|        DCA|       3492|     166.02|        133|        282|
|2014|        DEN|       3485|     119.12|         97|        237|
|2014|        ATL|       3344|     121.32|         98|        

In [43]:
if DATA_DIR is not None:
    PROJECT_OUTPUT = str(Path.cwd() / "flight_clean_stats.parquet")

    (
        flight_stats.write
        .mode("overwrite")
        .parquet(PROJECT_OUTPUT)
    )

    print("Đã ghi:", PROJECT_OUTPUT)

Đã ghi: /content/flight_clean_stats.parquet


## Bài tập về nhà

### Bài 1 - Schema

Tạo schema cho dữ liệu:

| product_id | product_name | price | quantity |
|---|---|---:|---:|

Yêu cầu:

- `product_id`: Integer
- `product_name`: String
- `price`: Double
- `quantity`: Integer

---

### Bài 2 - Cleaning

Với `DallasCouncilVoters.csv.gz`:

1. loại `VOTER_NAME` null;
2. trim khoảng trắng;
3. đổi tên thành uppercase;
4. tách tên thành array;
5. đếm số thành phần của tên.

---

### Bài 3 - Conditional Column

Tạo cột `flight_type`:

- `< 120 phút`: `Short`
- `120-240 phút`: `Medium`
- `> 240 phút`: `Long`

---

### Bài 4 - Parquet

1. đọc 4 file chuyến bay;
2. union;
3. ghi Parquet;
4. đọc lại;
5. so sánh schema và số dòng.

---

### Bài 5 - Cache

Thực hiện cùng một phép `groupBy()`:

1. trước khi cache;
2. sau khi cache;
3. `unpersist()` sau khi dùng.

Quan sát thời gian chạy.

---

### Bài 6 - Broadcast Join

Tạo bảng tra cứu mã airport nhỏ và join với dữ liệu chuyến bay.

So sánh:

```python
df.join(lookup_df, ...)
```

và

```python
df.join(F.broadcast(lookup_df), ...)
```

Dùng `explain()` để quan sát kế hoạch.

---

### Bài 7 - Validation

Với Dallas Council Votes:

1. xác định danh sách `VOTE CAST`;
2. tạo danh sách giá trị hợp lệ;
3. dùng join để giữ valid rows;
4. dùng `left_anti` để lấy invalid rows.

# BÀI LÀM
## Bài 1

In [1]:
from pyspark.sql.types import (
    StructType, StructField,
    IntegerType, StringType, DoubleType
)

product_schema = StructType([
    StructField("product_id", IntegerType(), True),
    StructField("product_name", StringType(), True),
    StructField("price", DoubleType(), True),
    StructField("quantity", IntegerType(), True)
])

## Bài 2

In [34]:
from pyspark.sql import functions as F

# Đọc dữ liệu
voter_df = spark.read.csv(
    str(DATA_DIR / "DallasCouncilVoters.csv.gz"),
    header=True,
    inferSchema=True
)

# Làm sạch tên
voter_clean = (
    voter_df
    .filter(F.col("VOTER_NAME").isNotNull())
    # Chuẩn hóa nhiều khoảng trắng thành một, rồi trim
    .withColumn(
        "VOTER_NAME",
        F.trim(F.regexp_replace("VOTER_NAME", r"\s+", " "))
    )
    .withColumn("VOTER_NAME", F.upper("VOTER_NAME"))
    .withColumn(
        "NAME_PARTS",
        F.filter(
            F.split("VOTER_NAME", " "),
            lambda word: word != ""
        )
    )
    .withColumn("NAME_WORD_COUNT", F.size("NAME_PARTS"))
)

voter_clean.select(
    "VOTER_NAME",
    "NAME_PARTS",
    "NAME_WORD_COUNT"
).show(10, truncate=False)

+-------------------+-----------------------+---------------+
|VOTER_NAME         |NAME_PARTS             |NAME_WORD_COUNT|
+-------------------+-----------------------+---------------+
|JENNIFER S. GATES  |[JENNIFER, S., GATES]  |3              |
|PHILIP T. KINGSTON |[PHILIP, T., KINGSTON] |3              |
|MICHAEL S. RAWLINGS|[MICHAEL, S., RAWLINGS]|3              |
|ADAM MEDRANO       |[ADAM, MEDRANO]        |2              |
|CASEY THOMAS       |[CASEY, THOMAS]        |2              |
|CAROLYN KING ARNOLD|[CAROLYN, KING, ARNOLD]|3              |
|SCOTT GRIGGS       |[SCOTT, GRIGGS]        |2              |
|B. ADAM MCGOUGH    |[B., ADAM, MCGOUGH]    |3              |
|LEE KLEINMAN       |[LEE, KLEINMAN]        |2              |
|SANDY GREYSON      |[SANDY, GREYSON]       |2              |
+-------------------+-----------------------+---------------+
only showing top 10 rows



### Bài 3

In [ ]:
from pyspark.sql import functions as F

exercise_flight = clean_flights.withColumn(
    "flight_type",
    F.when(
        F.col("elapsed_minutes").isNull() |
        (F.col("elapsed_minutes") <= 0),
        "Invalid"
    )
    .when(F.col("elapsed_minutes") < 120, "Short")
    .when(F.col("elapsed_minutes") <= 240, "Medium")
    .otherwise("Long")
)

exercise_flight.select(
    "destination", "elapsed_minutes", "flight_type"
).show(10, truncate=False)

# Đếm số chuyến theo loại
exercise_flight.groupBy("flight_type").count().orderBy(
    "flight_type"
).show()

## Bài 4

In [ ]:
from pathlib import Path
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField, StringType, IntegerType
)

flight_schema = StructType([
    StructField("Date (MM/DD/YYYY)", StringType(), True),
    StructField("Flight Number", StringType(), True),
    StructField("Destination Airport", StringType(), True),
    StructField("Actual elapsed time (Minutes)", IntegerType(), True)
])

# 1. Đọc 4 file bằng cùng schema
flight_dfs = []

for year in range(2014, 2018):
    file_path = DATA_DIR / f"AA_DFW_{year}_Departures_Short.csv.gz"

    year_df = spark.read.csv(
        str(file_path),
        header=True,
        schema=flight_schema
    )

    flight_dfs.append(year_df)

# 2. Union theo tên cột
flights_all = flight_dfs[0]

for year_df in flight_dfs[1:]:
    flights_all = flights_all.unionByName(year_df)

# 3. Ghi Parquet
parquet_path = str(
    (Path.cwd() / "output" / "AA_DFW_ALL.parquet").resolve()
)

flights_all.write.mode("overwrite").parquet(parquet_path)

# 4. Đọc lại bằng Spark
flights_parquet = spark.read.parquet(parquet_path)

# 5. So sánh schema và số dòng
before_count = flights_all.count()
after_count = flights_parquet.count()

print("Schema trước khi ghi:")
flights_all.printSchema()

print("Schema sau khi đọc lại:")
flights_parquet.printSchema()

print("Số dòng trước:", before_count)
print("Số dòng sau  :", after_count)

same_types = flights_all.dtypes == flights_parquet.dtypes
same_schema = flights_all.schema == flights_parquet.schema

print("Tên và kiểu cột giống nhau:", same_types)
print("Schema đầy đủ giống nhau :", same_schema)
print("Số dòng giống nhau       :", before_count == after_count)

assert before_count == after_count, "Số dòng không khớp!"
assert same_types, "Tên hoặc kiểu dữ liệu không khớp!"

## Bài 5

In [ ]:
from time import perf_counter

# Loại cache còn từ các bài trước để phép đo rõ ràng hơn
spark.catalog.clearCache()

# Đọc từ Parquet đã tạo ở Bài 4
benchmark_df = spark.read.parquet(parquet_path)

def run_groupby(frame):
    start = perf_counter()

    # collect chỉ đưa bảng tổng hợp nhỏ theo airport về driver.
    # Action này buộc Spark thực thi groupBy.
    result = (
        frame
        .groupBy("Destination Airport")
        .agg(
            F.count("*").alias("num_flights"),
            F.avg("Actual elapsed time (Minutes)").alias("avg_minutes")
        )
        .orderBy("Destination Airport")
        .collect()
    )

    return result, perf_counter() - start

# 1. Trước khi cache
before_result, before_time = run_groupby(benchmark_df)

# 2. Cache và materialize
benchmark_df.cache()

try:
    start = perf_counter()
    benchmark_df.count()
    materialize_time = perf_counter() - start

    after_result, after_time = run_groupby(benchmark_df)

    print(f"GroupBy trước cache : {before_time:.4f} giây")
    print(f"Materialize cache   : {materialize_time:.4f} giây")
    print(f"GroupBy sau cache   : {after_time:.4f} giây")

    print(
        "Số chuyến theo airport khớp:",
        [(r["Destination Airport"], r["num_flights"])
         for r in before_result]
        ==
        [(r["Destination Airport"], r["num_flights"])
         for r in after_result]
    )

    if after_time < before_time:
        print("Lần đo này: groupBy sau cache nhanh hơn.")
    else:
        print("Lần đo này: groupBy sau cache chưa nhanh hơn.")

finally:
    # 3. Giải phóng cache kể cả khi có lỗi
    benchmark_df.unpersist(blocking=True)

print("Còn cache:", benchmark_df.is_cached)

## Bài 6

In [ ]:
# Tạo bảng tra cứu nhỏ
lookup_df = spark.sql("""
    SELECT *
    FROM VALUES
        ('HNL', 'Honolulu'),
        ('OGG', 'Kahului'),
        ('SFO', 'San Francisco'),
        ('LAS', 'Las Vegas'),
        ('DTW', 'Detroit')
    AS airports(airport_code, airport_name)
""")

flight_join_df = flights_parquet.withColumn(
    "airport_code",
    F.upper(F.trim(F.col("Destination Airport")))
)

# Lưu cấu hình hiện tại
config_keys = [
    "spark.sql.autoBroadcastJoinThreshold",
    "spark.sql.adaptive.autoBroadcastJoinThreshold"
]

old_configs = {
    key: spark.conf.get(key, None)
    for key in config_keys
}

try:
    # Tạm tắt auto broadcast để nhìn rõ khác biệt.
    # Hint F.broadcast vẫn được áp dụng.
    for key in config_keys:
        spark.conf.set(key, "-1")

    normal_join = flight_join_df.join(
        lookup_df,
        on="airport_code",
        how="left"
    )

    broadcast_join = flight_join_df.join(
        F.broadcast(lookup_df),
        on="airport_code",
        how="left"
    )

    print("KẾ HOẠCH JOIN THƯỜNG")
    normal_join.explain(mode="formatted")

    print("\nKẾ HOẠCH BROADCAST JOIN")
    broadcast_join.explain(mode="formatted")

    print("\nKết quả join thường:")
    normal_join.select(
        "airport_code", "airport_name"
    ).show(10, truncate=False)

    print("Kết quả broadcast join:")
    broadcast_join.select(
        "airport_code", "airport_name"
    ).show(10, truncate=False)

    source_count = flight_join_df.count()
    normal_count = normal_join.count()
    broadcast_count = broadcast_join.count()

    print("Số dòng nguồn    :", source_count)
    print("Join thường      :", normal_count)
    print("Broadcast join   :", broadcast_count)

    assert source_count == normal_count == broadcast_count

finally:
    # Khôi phục cấu hình sau thí nghiệm
    for key, value in old_configs.items():
        if value is None:
            spark.conf.unset(key)
        else:
            spark.conf.set(key, value)

## Bài 7

In [ ]:
# Đọc dữ liệu, hỗ trợ trường CSV có xuống dòng trong dấu quote
votes_df = spark.read.csv(
    str(DATA_DIR / "DallasCouncilVotes.csv.gz"),
    header=True,
    multiLine=True,
    inferSchema=False
)

# Giữ giá trị gốc và tạo cột mã đã chuẩn hóa
votes_check = (
    votes_df
    .withColumn("VOTE_CAST_RAW", F.col("VOTE CAST"))
    .withColumn(
        "VOTE_CAST",
        F.upper(
            F.trim(
                F.regexp_replace(
                    F.col("VOTE CAST"),
                    r"\s+",
                    " "
                )
            )
        )
    )
)

# 1. Quan sát các giá trị đang có
print("Các giá trị VOTE CAST và số lần xuất hiện:")

(
    votes_check
    .groupBy("VOTE_CAST")
    .count()
    .orderBy(F.desc("count"))
    .show(50, truncate=100)
)

# 2. Whitelist thực hành.
# Cần đối chiếu với từ điển dữ liệu nếu áp dụng thực tế.
valid_values = [
    "YES", "NO", "N/A",
    "AWVT", "ABSNT", "ABSNT_CB", "ABST"
]

valid_schema = StructType([
    StructField("VOTE_CAST", StringType(), True)
])

valid_values_df = spark.createDataFrame(
    [(value,) for value in valid_values],
    schema=valid_schema
).dropDuplicates()

# 3. Giữ các dòng khớp whitelist
valid_rows = votes_check.join(
    F.broadcast(valid_values_df),
    on="VOTE_CAST",
    how="inner"
)

# 4. Lấy các dòng không khớp, bao gồm NULL/rỗng
invalid_rows = votes_check.join(
    F.broadcast(valid_values_df),
    on="VOTE_CAST",
    how="left_anti"
)

before = votes_check.count()
valid_count = valid_rows.count()
invalid_count = invalid_rows.count()

print("Tổng số dòng :", before)
print("Valid rows  :", valid_count)
print("Invalid rows:", invalid_count)

assert before == valid_count + invalid_count, \
    "Tổng valid và invalid không khớp nguồn!"

print("\nThống kê mã hợp lệ:")
valid_rows.groupBy("VOTE_CAST").count().orderBy(
    F.desc("count")
).show()

print("Ví dụ dòng không hợp lệ:")
invalid_rows.select(
    "VOTE_CAST_RAW", "VOTE_CAST"
).show(10, truncate=100)

## Tổng kết

Trình tự nội dung của notebook:

```text
Data Cleaning
   ↓
Schema
   ↓
Immutability + Lazy Processing
   ↓
Parquet
   ↓
DataFrame Column Operations
   ↓
when / otherwise
   ↓
UDF
   ↓
Partition + ID
   ↓
Cache
   ↓
Import Performance
   ↓
Execution Plan / Shuffle / Broadcast
   ↓
Data Pipeline
   ↓
Validation
   ↓
Clean Output
```

## Các nguyên tắc quan trọng

1. Định nghĩa schema rõ ràng khi biết trước cấu trúc dữ liệu.
2. Nhớ rằng DataFrame là immutable và Spark thực thi lazy.
3. Ưu tiên built-in Spark functions trước Python UDF.
4. Tránh shuffle không cần thiết.
5. Broadcast bảng nhỏ khi join nếu phù hợp.
6. Chỉ cache khi DataFrame thực sự được tái sử dụng.
7. Dùng Parquet cho dữ liệu trung gian/phân tích khi phù hợp.
8. Data cleaning cần đi kèm validation.
9. Nên lưu riêng dữ liệu invalid để kiểm tra thay vì xóa âm thầm.
10. Dùng `explain()` để hiểu kế hoạch Spark thực thi.

In [ ]:
# Khi hoàn thành toàn bộ notebook có thể giải phóng tài nguyên:
spark.stop()